# 06 — Metadata Inspection

Every Iceberg table exposes its internals as queryable metadata tables via
`table.inspect.*`. Each call returns a PyArrow `Table` — convert to Polars for analysis.

This is the reference notebook the rest of the curriculum links to.

**Prerequisites:** Run 01–05 to populate snapshots, files, and refs.

In [1]:
import polars as pl

import os

from pyiceberg.catalog.rest import RestCatalog

NESSIE_URI       = os.environ["NESSIE_URI"]
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

catalog = RestCatalog(
    name="nessie",
    **{
        "uri": NESSIE_URI,
        "warehouse": WAREHOUSE_URI,
        "s3.endpoint": S3_ENDPOINT,
        "s3.access-key-id": S3_ACCESS_KEY,
        "s3.secret-access-key": S3_SECRET_KEY,
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)

print("Connected to catalog:", catalog.name)

table = catalog.load_table(("demo", "events"))
table.refresh()
print("Table:", table.name())

Connected to catalog: nessie


Table: ('demo', 'events')


## 1. `inspect.snapshots()` — every snapshot the current metadata knows about

In [2]:
snapshots = pl.from_arrow(table.inspect.snapshots())
print(f"Snapshots: {len(snapshots)}")
snapshots

Snapshots: 1


committed_at,snapshot_id,parent_id,operation,manifest_list,summary
datetime[ms],i64,i64,str,str,list[struct[2]]
2026-05-15 11:06:37.249,6746404416189507434,null,"""append""","""s3://iceberg-warehouse/warehou…","[{""added-files-size"",""2466""}, {""added-data-files"",""1""}, … {""total-equality-deletes"",""0""}]"


## 2. `inspect.files()` — every data file in the current snapshot

In [3]:
files = pl.from_arrow(table.inspect.files())
print(f"Files: {len(files)}")
files.select(["file_path", "file_format", "record_count", "file_size_in_bytes"])

Files: 9


file_path,file_format,record_count,file_size_in_bytes
str,cat,i64,i64
"""s3://iceberg-warehouse/warehou…","""PARQUET""",1,2466
"""s3://iceberg-warehouse/warehou…","""PARQUET""",1,2466
"""s3://iceberg-warehouse/warehou…","""PARQUET""",1,2487
"""s3://iceberg-warehouse/warehou…","""PARQUET""",1,2459
"""s3://iceberg-warehouse/warehou…","""PARQUET""",2,2489
"""s3://iceberg-warehouse/warehou…","""PARQUET""",1,2466
"""s3://iceberg-warehouse/warehou…","""PARQUET""",1,2459
"""s3://iceberg-warehouse/warehou…","""PARQUET""",2,2521
"""s3://iceberg-warehouse/warehou…","""PARQUET""",1,2459


## 3. `inspect.manifests()` — manifest files (one level above data files)

In [4]:
manifests = pl.from_arrow(table.inspect.manifests())
print(f"Manifests: {len(manifests)}")
manifests

Manifests: 6


content,path,length,partition_spec_id,added_snapshot_id,added_data_files_count,existing_data_files_count,deleted_data_files_count,added_delete_files_count,existing_delete_files_count,deleted_delete_files_count,partition_summaries
i8,str,i64,i32,i64,i32,i32,i32,i32,i32,i32,list[struct[4]]
0,"""s3://iceberg-warehouse/warehou…",4750,1,6746404416189507434,1,0,0,0,0,0,"[{false,false,""2024-03-02-12"",""2024-03-02-12""}]"
0,"""s3://iceberg-warehouse/warehou…",4750,1,5577363523623265553,1,0,0,0,0,0,"[{false,false,""2024-01-18-09"",""2024-01-18-09""}]"
0,"""s3://iceberg-warehouse/warehou…",5108,1,1380997151782029379,2,0,0,0,0,0,"[{false,false,""2024-03-01-10"",""2024-03-01-11""}]"
0,"""s3://iceberg-warehouse/warehou…",5105,1,7176660290452338699,2,0,0,0,0,0,"[{false,false,""2024-02-01-09"",""2024-02-01-10""}]"
0,"""s3://iceberg-warehouse/warehou…",4774,0,4874759031295520682,1,0,0,0,0,0,"[{false,false,""2024-01-15"",""2024-01-15""}]"
0,"""s3://iceberg-warehouse/warehou…",5133,0,3847490575672181232,0,2,0,0,0,0,"[{false,false,""2024-01-16"",""2024-01-17""}]"


## 4. `inspect.history()` — operation log

In [5]:
history = pl.from_arrow(table.inspect.history())
history

made_current_at,snapshot_id,parent_id,is_current_ancestor
datetime[ms],i64,i64,bool
2026-05-15 11:06:37.249,6746404416189507434,null,true


## 5. `inspect.partitions()` — current partition layout

In [6]:
partitions = pl.from_arrow(table.inspect.partitions())
partitions

partition,spec_id,record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
struct[2],i32,i64,i32,i64,i64,i32,i64,i32,datetime[ms],i64
"{null,474828}",1,1,1,2466,0,0,0,0,2026-05-15 11:06:37.249,6746404416189507434
"{null,473769}",1,1,1,2466,0,0,0,0,null,null
"{null,474802}",1,1,1,2487,0,0,0,0,null,null
"{null,474803}",1,1,1,2459,0,0,0,0,null,null
"{null,474105}",1,2,1,2489,0,0,0,0,null,null
"{null,474106}",1,1,1,2466,0,0,0,0,null,null
"{2024-01-15,null}",0,1,1,2459,0,0,0,0,null,null
"{2024-01-16,null}",0,2,1,2521,0,0,0,0,null,null
"{2024-01-17,null}",0,1,1,2459,0,0,0,0,null,null


## 6. `inspect.refs()` — Iceberg-native branches and tags

Not to be confused with Nessie catalog refs (see notebook 08). These are
stored inside the table's own metadata.

In [7]:
refs = pl.from_arrow(table.inspect.refs())
refs

name,type,snapshot_id,max_reference_age_in_ms,min_snapshots_to_keep,max_snapshot_age_in_ms
str,cat,i64,i64,i32,i64
"""main""","""BRANCH""",6746404416189507434,null,null,null


---
**Cross-references:**
- [07_iceberg_branches_tags.ipynb](07_iceberg_branches_tags.ipynb) creates entries that show up in `inspect.refs()`.
- [09_snapshot_management.ipynb](09_snapshot_management.ipynb) uses `inspect.snapshots()` to pick rollback targets.
- [10_spark_maintenance.ipynb](10_spark_maintenance.ipynb) compares `inspect.files()` counts before and after compaction.